# NMF on ModulAir 00683

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-000689
data = pd.read_csv('MOD-00689.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:05Z,577612306,2025-12-31T18:59:05Z,MOD-00689,49.6,0.5,6.743,0.692,0.187,0.038,0.031,...,29.938,38.903,14340,14341,14342,14476.0,14501.0,14551.0,14526.0,0.90
2025-12-31T23:58:05Z,577610344,2025-12-31T18:58:05Z,MOD-00689,49.7,0.5,7.506,0.821,0.251,0.027,0.042,...,29.223,39.248,14340,14341,14342,14476.0,14501.0,14551.0,14526.0,0.97
2025-12-31T23:57:05Z,577610343,2025-12-31T18:57:05Z,MOD-00689,49.8,0.4,7.911,0.835,0.231,0.051,0.034,...,29.449,39.248,14340,14341,14342,14476.0,14501.0,14551.0,14526.0,1.61
2025-12-31T23:56:05Z,577610342,2025-12-31T18:56:05Z,MOD-00689,50.0,0.4,7.611,0.901,0.256,0.050,0.030,...,29.677,38.185,14340,14341,14342,14476.0,14501.0,14551.0,14526.0,1.54
2025-12-31T23:55:05Z,577610341,2025-12-31T18:55:05Z,MOD-00689,49.9,0.4,8.115,0.936,0.324,0.078,0.046,...,30.155,38.190,14340,14341,14342,14476.0,14501.0,14551.0,14526.0,1.50


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:05Z,2025-12-31T18:59:05Z,775.946,2.402,29.938,38.903,6.743,0.692,0.187,0.038,0.031,0.020
2025-12-31T23:58:05Z,2025-12-31T18:58:05Z,768.897,2.401,29.223,39.248,7.506,0.821,0.251,0.027,0.042,0.015
2025-12-31T23:57:05Z,2025-12-31T18:57:05Z,760.097,2.480,29.449,39.248,7.911,0.835,0.231,0.051,0.034,0.015
2025-12-31T23:56:05Z,2025-12-31T18:56:05Z,784.175,2.519,29.677,38.185,7.611,0.901,0.256,0.050,0.030,0.010
2025-12-31T23:55:05Z,2025-12-31T18:55:05Z,774.804,2.519,30.155,38.190,8.115,0.936,0.324,0.078,0.046,0.012


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:05Z,775.946,2.402,29.938,38.903,6.743,0.692,0.187,0.038,0.031,0.020
1,2025-12-31T18:58:05Z,768.897,2.401,29.223,39.248,7.506,0.821,0.251,0.027,0.042,0.015
2,2025-12-31T18:57:05Z,760.097,2.480,29.449,39.248,7.911,0.835,0.231,0.051,0.034,0.015
3,2025-12-31T18:56:05Z,784.175,2.519,29.677,38.185,7.611,0.901,0.256,0.050,0.030,0.010
4,2025-12-31T18:55:05Z,774.804,2.519,30.155,38.190,8.115,0.936,0.324,0.078,0.046,0.012


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:05,775.946,2.402,29.938,38.903,6.743,0.692,0.187,0.038,0.031,0.020
1,2025-12-31 18:58:05,768.897,2.401,29.223,39.248,7.506,0.821,0.251,0.027,0.042,0.015
2,2025-12-31 18:57:05,760.097,2.480,29.449,39.248,7.911,0.835,0.231,0.051,0.034,0.015
3,2025-12-31 18:56:05,784.175,2.519,29.677,38.185,7.611,0.901,0.256,0.050,0.030,0.010
4,2025-12-31 18:55:05,774.804,2.519,30.155,38.190,8.115,0.936,0.324,0.078,0.046,0.012


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-08-07 03:00:00,723.72,11.38,39.37,1.73,11.69,1.54,0.30,0.05,0.05,0.04
1,2025-08-07 04:00:00,739.99,10.25,36.40,1.85,11.60,1.53,0.31,0.04,0.05,0.04
2,2025-08-07 05:00:00,783.80,14.11,31.41,1.91,10.35,1.27,0.26,0.05,0.06,0.05
3,2025-08-07 06:00:00,856.33,20.79,31.60,1.84,17.28,2.46,0.46,0.08,0.09,0.07
4,2025-08-07 07:00:00,1584.73,19.63,37.99,2.69,43.08,8.03,1.33,0.17,0.16,0.11
...,...,...,...,...,...,...,...,...,...,...,...
3394,2025-12-31 14:00:00,701.00,28.14,46.26,2.02,7.36,0.59,0.18,0.03,0.03,0.01
3395,2025-12-31 15:00:00,725.34,28.23,45.66,1.94,7.07,0.60,0.18,0.03,0.03,0.01
3396,2025-12-31 16:00:00,732.31,28.74,43.74,2.43,7.85,0.69,0.21,0.03,0.03,0.02
3397,2025-12-31 17:00:00,761.58,29.63,40.06,2.38,8.35,0.82,0.25,0.04,0.03,0.02


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-08-07 03:00:00,723.72,11.38,39.37,1.73,11.69,1.54,0.30,0.05,0.05,0.04
2025-08-07 04:00:00,739.99,10.25,36.40,1.85,11.60,1.53,0.31,0.04,0.05,0.04
2025-08-07 05:00:00,783.80,14.11,31.41,1.91,10.35,1.27,0.26,0.05,0.06,0.05
2025-08-07 06:00:00,856.33,20.79,31.60,1.84,17.28,2.46,0.46,0.08,0.09,0.07
2025-08-07 07:00:00,1584.73,19.63,37.99,2.69,43.08,8.03,1.33,0.17,0.16,0.11


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-08-07 03:00:00,0.089636,0.271859,0.514708,0.006620,0.233847,0.075159,0.038911,0.032895,0.025510,0.023256
2025-08-07 04:00:00,0.091651,0.244864,0.475879,0.007079,0.232046,0.074671,0.040208,0.026316,0.025510,0.023256
2025-08-07 05:00:00,0.097077,0.337076,0.410642,0.007308,0.207041,0.061981,0.033722,0.032895,0.030612,0.029070
2025-08-07 06:00:00,0.106060,0.496656,0.413126,0.007041,0.345669,0.120059,0.059663,0.052632,0.045918,0.040698
2025-08-07 07:00:00,0.196275,0.468944,0.496666,0.010293,0.861772,0.391898,0.172503,0.111842,0.081633,0.063953


In [11]:
data.to_csv(r'MOD-000689_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-08-07 03:00:00,0.079581,0.272408,0.515665,0.007828,0.235439,0.070515,0.040617,0.033010,0.026598,0.023783
2025-08-07 04:00:00,0.073912,0.245821,0.477757,0.007184,0.233728,0.070316,0.040006,0.031593,0.024561,0.021865
2025-08-07 05:00:00,0.071769,0.338498,0.413694,0.006846,0.205752,0.065119,0.040009,0.035316,0.027811,0.024270
2025-08-07 06:00:00,0.086941,0.497654,0.415550,0.007363,0.345345,0.116039,0.070865,0.059556,0.040213,0.033713
2025-08-07 07:00:00,0.116692,0.473167,0.503325,0.007330,0.887368,0.309225,0.183609,0.141928,0.083323,0.069540
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.107930,0.671129,0.602271,0.011093,0.146652,0.032637,0.016859,0.015095,0.015437,0.012575
2025-12-31 15:00:00,0.107123,0.673499,0.594762,0.011016,0.141533,0.031491,0.016618,0.015430,0.015903,0.012995
2025-12-31 16:00:00,0.105856,0.685773,0.570077,0.010712,0.156398,0.037650,0.020429,0.018332,0.016917,0.013615


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.029791,0.060124,0.084739,0.008821
1,0.026371,0.060469,0.079114,0.008020
2,0.038773,0.052930,0.053645,0.012049
3,0.055508,0.094763,0.033777,0.020803
4,0.038064,0.257959,0.051018,0.047260
...,...,...,...,...
3323,0.083808,0.029839,0.058806,0.001054
3324,0.084212,0.028428,0.056841,0.001564
3325,0.085342,0.033406,0.049513,0.002746
3326,0.087712,0.037521,0.035835,0.005140


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,0.882829,7.777061,3.973823,0.090702,0.361704,0.000000,0.000000,0.045070,0.070933,0.041793
Feature 2,0.209876,0.639273,0.480845,0.001972,3.329440,1.068698,0.529966,0.266728,0.030733,0.000000
Feature 3,0.468770,0.000000,4.331637,0.058215,0.288951,0.000000,0.000000,0.030623,0.120578,0.131039
Feature 4,0.106482,0.258826,0.148833,0.008438,0.000000,0.709803,0.992370,1.477898,1.408028,1.296309


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-08-07 03:00:00,0.029791,0.060124,0.084739,0.008821
2025-08-07 04:00:00,0.026371,0.060469,0.079114,0.008020
2025-08-07 05:00:00,0.038773,0.052930,0.053645,0.012049
2025-08-07 06:00:00,0.055508,0.094763,0.033777,0.020803
2025-08-07 07:00:00,0.038064,0.257959,0.051018,0.047260
...,...,...,...,...
2025-12-31 14:00:00,0.083808,0.029839,0.058806,0.001054
2025-12-31 15:00:00,0.084212,0.028428,0.056841,0.001564
2025-12-31 16:00:00,0.085342,0.033406,0.049513,0.002746


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,0.882829,7.777061,3.973823,0.090702,0.361704,0.000000,0.000000,0.045070,0.070933,0.041793
Factor 2,0.209876,0.639273,0.480845,0.001972,3.329440,1.068698,0.529966,0.266728,0.030733,0.000000
Factor 3,0.468770,0.000000,4.331637,0.058215,0.288951,0.000000,0.000000,0.030623,0.120578,0.131039
Factor 4,0.106482,0.258826,0.148833,0.008438,0.000000,0.709803,0.992370,1.477898,1.408028,1.296309


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.535692,0.077552,0.356172,0.019285,0.011298
no,0.542956,0.007190,0.436357,0.015076,-0.001579
no2,0.943543,0.047231,0.000000,0.009373,-0.000146
o3,0.408316,0.030088,0.557316,0.004565,-0.000284
bin0,0.131454,0.736866,0.131495,0.000000,0.000186
bin1,0.000000,0.824934,0.000000,0.268548,-0.093482
bin2,0.000000,0.535046,0.000000,0.491062,-0.026109
bin3,0.065536,0.236188,0.055757,0.641437,0.001083
bin4,0.107049,0.028244,0.227858,0.634248,0.002601
bin5,0.070676,0.000000,0.277484,0.654333,-0.002494


In [20]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')